In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold

In [2]:
# Load the dataset
data = pd.read_csv("../data/BCF_model_dataset.csv")  # Replace 'your_file.csv' with your actual file path
data

,Chem ID,SMILES,CAS,CID,InChIKey,logBCF
0,1,CCN(CC)c1ccc2c(-c3ccccc3C(=O)O)c3ccc(N(CC)CC)c...,64381-98-2,6695,CVAVMIODJQHEEH-UHFFFAOYSA-O,-0.70
1,2,O=C(O)c1ccc(Cl)c(Cl)c1,51-44-5,5817,VPHHJAOJUJHJKD-UHFFFAOYSA-N,0.35
2,3,Brc1cc(Br)c(-c2c(Br)cc(Br)cc2Br)c(Br)c1,59261-08-4,93322,LNFYSRMCCKKDEH-UHFFFAOYSA-N,4.27
3,4,CC(C)Oc1cccc(NC(=O)c2ccccc2C(F)(F)F)c1,66332-96-5,47898,PTCGDEVVHUXTMP-UHFFFAOYSA-N,1.50
4,5,C1=CCCC=CCCC=CCC1,4904-61-4,12668,ZOLLIQAKMYWTBR-UHFFFAOYSA-N,3.82
...,...,...,...,...,...,...
1667,1668,CC(C)c1ccc2c(c1)CC[C@H]1[C@](C)(C(=O)O)CCC[C@]21C,1740-19-8,94391,NFWKVWVWBFBAOV-MISYRCLQSA-N,2.90
1668,1669,CC12CCC(CC1)C(C)(C)O2,470-82-6,2758,WEEGYLXZBRQIMU-UHFFFAOYSA-N,0.48
1669,1670,CC(Cl)(CCl)OP(=O)(OC(C)(Cl)CCl)OC(C)(Cl)CCl,NaN,14968365,YQKGJRGUAQVYNL-UHFFFAOYSA-N,0.52
1670,1671,CCCCCCCCC(Br)CBr,28467-71-2,20052,XBRBOTTWTQOCJH-UHFFFAOYSA-N,2.51


In [3]:
# Convert SMILES to RDKit Molecule objects
data['mol'] = data['SMILES'].apply(Chem.MolFromSmiles)
data

,Chem ID,SMILES,CAS,CID,InChIKey,logBCF,mol
0,1,CCN(CC)c1ccc2c(-c3ccccc3C(=O)O)c3ccc(N(CC)CC)c...,64381-98-2,6695,CVAVMIODJQHEEH-UHFFFAOYSA-O,-0.70,<rdkit.Chem.rdchem.Mol object at 0x00000214EB2...
1,2,O=C(O)c1ccc(Cl)c(Cl)c1,51-44-5,5817,VPHHJAOJUJHJKD-UHFFFAOYSA-N,0.35,<rdkit.Chem.rdchem.Mol object at 0x00000214EB2...
2,3,Brc1cc(Br)c(-c2c(Br)cc(Br)cc2Br)c(Br)c1,59261-08-4,93322,LNFYSRMCCKKDEH-UHFFFAOYSA-N,4.27,<rdkit.Chem.rdchem.Mol object at 0x00000214EB2...
3,4,CC(C)Oc1cccc(NC(=O)c2ccccc2C(F)(F)F)c1,66332-96-5,47898,PTCGDEVVHUXTMP-UHFFFAOYSA-N,1.50,<rdkit.Chem.rdchem.Mol object at 0x00000214EB2...
4,5,C1=CCCC=CCCC=CCC1,4904-61-4,12668,ZOLLIQAKMYWTBR-UHFFFAOYSA-N,3.82,<rdkit.Chem.rdchem.Mol object at 0x00000214EB2...
...,...,...,...,...,...,...,...
1667,1668,CC(C)c1ccc2c(c1)CC[C@H]1[C@](C)(C(=O)O)CCC[C@]21C,1740-19-8,94391,NFWKVWVWBFBAOV-MISYRCLQSA-N,2.90,<rdkit.Chem.rdchem.Mol object at 0x00000214EB3...
1668,1669,CC12CCC(CC1)C(C)(C)O2,470-82-6,2758,WEEGYLXZBRQIMU-UHFFFAOYSA-N,0.48,<rdkit.Chem.rdchem.Mol object at 0x00000214EB3...
1669,1670,CC(Cl)(CCl)OP(=O)(OC(C)(Cl)CCl)OC(C)(Cl)CCl,NaN,14968365,YQKGJRGUAQVYNL-UHFFFAOYSA-N,0.52,<rdkit.Chem.rdchem.Mol object at 0x00000214EB3...
1670,1671,CCCCCCCCC(Br)CBr,28467-71-2,20052,XBRBOTTWTQOCJH-UHFFFAOYSA-N,2.51,<rdkit.Chem.rdchem.Mol object at 0x00000214EB3...


In [4]:
# Get a list of all RDKit descriptor functions
descriptor_names = [desc_name for desc_name, _ in Descriptors.descList]

# Define a function to calculate all RDKit descriptors for a given molecule
def calculate_all_descriptors(mol):
    if mol is None:
        return pd.Series([None] * len(descriptor_names))
    return pd.Series([func(mol) for _, func in Descriptors.descList])

# Calculate descriptors for each molecule
desc_df = data['mol'].apply(calculate_all_descriptors)
desc_df.columns = descriptor_names

# Update the original data to match valid descriptors and concatenate
final_df = pd.concat([data.loc[desc_df.index, ['Chem ID', 'SMILES']], desc_df], axis=1)

# Save the final dataframe to a CSV file
final_df.to_csv("../data/BCF_model_dataset_all_descriptors.csv", index=False)

final_df

,Chem ID,SMILES,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,1,CCN(CC)c1ccc2c(-c3ccccc3C(=O)O)c3ccc(N(CC)CC)c...,12.072739,12.072739,0.284203,-0.937764,0.236446,11.151515,443.567,412.319,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,O=C(O)c1ccc(Cl)c(Cl)c1,10.362722,10.362722,0.138333,-1.010586,0.740184,9.636364,191.013,186.981,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,Brc1cc(Br)c(-c2c(Br)cc(Br)cc2Br)c(Br)c1,3.608104,3.608104,1.020808,1.020808,0.301499,10.777778,627.588,623.556,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,CC(C)Oc1cccc(NC(=O)c2ccccc2C(F)(F)F)c1,12.955037,12.955037,0.049160,-4.586771,0.881959,11.391304,323.314,307.186,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,C1=CCCC=CCCC=CCC1,2.291454,2.291454,1.208546,1.208546,0.472025,20.000000,162.276,144.132,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1667,1668,CC(C)c1ccc2c(c1)CC[C@H]1[C@](C)(C(=O)O)CCC[C@]21C,11.928632,11.928632,0.022304,-0.605856,0.842704,34.136364,300.442,272.218,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1668,1669,CC12CCC(CC1)C(C)(C)O2,6.060185,6.060185,0.157986,0.157986,0.521026,47.727273,154.253,136.109,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1669,1670,CC(Cl)(CCl)OP(=O)(OC(C)(Cl)CCl)OC(C)(Cl)CCl,12.640810,12.640810,0.189469,-4.282662,0.346100,21.850000,430.907,415.787,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1670,1671,CCCCCCCCC(Br)CBr,3.612765,3.612765,0.683958,0.683958,0.441818,13.250000,300.078,279.918,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0


In [6]:
# Load feature matrix X
X = final_df.drop(columns=['Chem ID', 'SMILES'])

# Preprocess the feature matrix

# Step 1: Remove columns with missing values
X_clean = X.dropna(axis=1)

# Step 2: Normalize the data
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X_clean)

# Step 3: Remove low variance features and retain the corresponding column names
variance_thresh = VarianceThreshold(threshold=0.01)  # Set variance threshold
X_variance_clean = variance_thresh.fit_transform(X_normalized)

# Get the names of the retained columns
retained_columns = X_clean.columns[variance_thresh.get_support()]

# Convert the processed data back to a DataFrame with column names
X_variance_clean_df = pd.DataFrame(X_variance_clean, columns=retained_columns)

# save results
X_variance_clean_df.to_csv("../data/BCF_model_dataset_90 descriptors.csv")
X_variance_clean_df

,MaxAbsEStateIndex,MaxEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,MaxPartialCharge,...,fr_allylic_oxid,fr_amide,fr_aryl_methyl,fr_benzene,fr_ester,fr_furan,fr_ketone,fr_nitro_arom,fr_phenol,fr_phenol_noOrthoHbond
0,0.779146,0.779146,0.762647,0.238716,0.095872,0.242011,0.231842,0.242063,0.271739,0.599940,...,0.000000,0.0,0.000000,0.272727,0.0,0.0,0.0,0.0,0.0,0.0
1,0.653128,0.653128,0.756763,0.782168,0.076246,0.085493,0.088426,0.084945,0.068841,0.560750,...,0.000000,0.0,0.000000,0.090909,0.0,0.0,0.0,0.0,0.0,0.0
2,0.155354,0.155354,0.920887,0.308898,0.091031,0.356056,0.366283,0.352676,0.134058,0.126863,...,0.000000,0.0,0.000000,0.181818,0.0,0.0,0.0,0.0,0.0,0.0
3,0.844166,0.844166,0.467829,0.935120,0.098978,0.167485,0.164930,0.167547,0.184783,0.675652,...,0.000000,0.2,0.000000,0.181818,0.0,0.0,0.0,0.0,0.0,0.0
4,0.058325,0.058325,0.936055,0.492867,0.210486,0.067683,0.061155,0.067689,0.083333,0.043265,...,0.857143,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1667,0.768526,0.768526,0.789463,0.892770,0.393594,0.153310,0.142675,0.153339,0.181159,0.523970,...,0.000000,0.0,0.166667,0.090909,0.0,0.0,0.0,0.0,0.0,0.0
1668,0.336058,0.336058,0.851177,0.545731,0.569636,0.062711,0.056049,0.062723,0.079710,0.181195,...,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1669,0.821010,0.821010,0.492399,0.357014,0.234449,0.234165,0.234049,0.232541,0.184783,0.764353,...,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1670,0.155698,0.155698,0.893672,0.460279,0.123053,0.153085,0.147576,0.151964,0.097826,0.122023,...,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
